# CBRP Methodologies Analysis - R Notebook

This notebook contains analysis for the CBRP (Capacitated Bicycle Routing Problem) methodologies.


In [2]:
# Load required libraries
library(scmamp)
library(readxl)
# install.packages("readxl")

## Data Loading

Load and prepare the data for analysis.


In [3]:
# Define path to the Excel file
excel_path <- "/home/carlos/Documentos/cbrp-methodologies/results-walk-models.xlsx"

# Get all sheet names
sheet_names <- excel_sheets(excel_path)

# Remove the "Compare-all" sheet if it exists
sheet_names <- sheet_names[!(sheet_names == "Compare-all" | grepl("Greedy", sheet_names, ignore.case = TRUE))]

# Remove the word "Trail" from each sheet name for use as column/list names
clean_names <- gsub("results-walk-", "", sheet_names)
clean_names <- trimws(clean_names)  # Remove any accidental leading/trailing spaces
# Rename method names in clean_names for consistency
clean_names <- gsub("^exp$", "Walk-CBRP", clean_names)
clean_names <- gsub("^exp-prep$", "Walk-CBRP-Prep", clean_names)
clean_names <- gsub("^exp-frac-cut$", "Walk-CBRP-Frac", clean_names)
clean_names <- gsub("^exp-frac-cut-prep$", "Walk-CBRP-Frac-Prep", clean_names)
clean_names <- gsub("^walk-mtz-results$", "Walk-CBRP-MTZ", clean_names)
clean_names <- gsub("^results-greedy-heuristic$", "Walk-Greedy", clean_names)
clean_names <- gsub("^results-greedy-heuristic-prep$", "Walk-Greedy-Prep", clean_names)

# Read each remaining sheet into a list of dataframes named by cleaned name
data_list <- lapply(sheet_names, function(sheet) {
  read_excel(excel_path, sheet = sheet)
})
names(data_list) <- clean_names
print(clean_names)


[1] "Walk-CBRP"           "Walk-CBRP-Frac"      "Walk-CBRP-Frac-Prep"
[4] "Walk-CBRP-Prep"      "Walk-CBRP-MTZ"      


## Exploratory Data Analysis

View and summarize the loaded data.


In [4]:

# Helper: coerce Excel-imported numeric columns that may come as character
# Handles comma decimals ("12,34") and thousands separators ("1.234,56").
excel_num <- function(x) {
  if (is.numeric(x)) return(x)
  x <- trimws(as.character(x))
  x[x == ""] <- NA

  has_dot <- grepl("\\.", x)
  has_comma <- grepl(",", x)

  y <- x
  both <- has_dot & has_comma
  y[both] <- gsub("\\.", "", y[both])      # remove thousands '.'
  y[has_comma] <- gsub(",", ".", y[has_comma]) # convert decimal ',' -> '.'

  suppressWarnings(as.numeric(y))
}

# IMPORTANT: sheets may have different row orders/filters.
# Build comparison data frames by aligning rows using the 'Instance' key.
instances_all <- sort(unique(unlist(lapply(data_list, function(df) df$Instance))))

LB_df <- data.frame(row.names = instances_all)
UB_df <- data.frame(row.names = instances_all)

for (method in names(data_list)) {
  df <- data_list[[method]]
  # named vectors keyed by Instance
  lb_map <- setNames(excel_num(df$LB), df$Instance)
  ub_map <- setNames(excel_num(df$UB), df$Instance)

  LB_df[[method]] <- unname(lb_map[instances_all])
  UB_df[[method]] <- unname(ub_map[instances_all])
}

# Fill missing instances (when a sheet doesn't contain some Instance rows)
LB_df[is.na(LB_df)] <- 0
UB_df[is.na(UB_df)] <- Inf

# Quick sanity check (optional): ensure numeric and consistent nrow
# stopifnot(nrow(LB_df) == length(instances_all), nrow(UB_df) == length(instances_all))
# str(LB_df); str(UB_df)


In [5]:
# Compute the CriticalValue for LB_df and UB_df using the qf function.
# Normally, for the Iman-Davenport test, the critical value is based on the F-distribution.

# For LB_df
k_LB <- ncol(LB_df)
N_LB <- nrow(LB_df)
CriticalValue_LB <- qf(0.95, k_LB - 1, (k_LB - 1) * (N_LB - 1))
print(paste("Critical Value for LB_df:", CriticalValue_LB))

# For UB_df
k_UB <- ncol(UB_df)
N_UB <- nrow(UB_df)
CriticalValue_UB <- qf(0.95, k_UB - 1, (k_UB - 1) * (N_UB - 1))
print(paste("Critical Value for UB_df:", CriticalValue_UB))


[1] "Critical Value for LB_df: 2.43116424264913"
[1] "Critical Value for UB_df: 2.43116424264913"


In [6]:
print("Lower Bound")
res_id_lb = imanDavenportTest(LB_df)
print(res_id_lb[['statistic']])

res_nemenyi_lb = nemenyiTest(LB_df)
print(res_nemenyi_lb[['statistic']])

[1] "Lower Bound"
Corrected Friedman's chi-squared 
                        9.527342 
Critical difference 
          0.9861445 


In [7]:
print("Upper Bound")
res_id_ub = imanDavenportTest(UB_df)
print(res_id_ub[['statistic']])

res_nemenyi_ub = nemenyiTest(UB_df)
print(res_nemenyi_ub[['statistic']])

[1] "Upper Bound"
Corrected Friedman's chi-squared 
                        17.95701 
Critical difference 
          0.9861445 


## Visualization

Create plots and visualizations of the data.


In [8]:
# ==== Plot for LB_df ====
N_LB <- nrow(LB_df)
k_LB <- ncol(LB_df)

ranks_matrix_LB <- apply(LB_df, 1, rank)
if (!is.matrix(ranks_matrix_LB)) ranks_matrix_LB <- matrix(ranks_matrix_LB, nrow = k_LB)
ranks_matrix_LB <- t(ranks_matrix_LB) # N x k

R_j_LB <- colMeans(ranks_matrix_LB)

approach_labels_LB <- colnames(LB_df)
for (j in 1:k_LB) {
  approach_labels_LB[j] <- sprintf("%s\n(%.2f)", colnames(LB_df)[j], R_j_LB[j])
}
LB_df_plot <- LB_df
colnames(LB_df_plot) <- approach_labels_LB

pdf("cd_walk_models_lb.pdf", width=12, height=6)
plotCD(LB_df_plot, cex=1, decreasing=FALSE)
dev.off()

# ==== Plot for UB_df ====
N_UB <- nrow(UB_df)
k_UB <- ncol(UB_df)

ranks_matrix_UB <- apply(UB_df, 1, rank)
if (!is.matrix(ranks_matrix_UB)) ranks_matrix_UB <- matrix(ranks_matrix_UB, nrow = k_UB)
ranks_matrix_UB <- t(ranks_matrix_UB) # N x k

R_j_UB <- colMeans(ranks_matrix_UB)

approach_labels_UB <- colnames(UB_df)
for (j in 1:k_UB) {
  approach_labels_UB[j] <- sprintf("%s\n(%.2f)", colnames(UB_df)[j], R_j_UB[j])
}
UB_df_plot <- UB_df
colnames(UB_df_plot) <- approach_labels_UB

pdf("cd_walk_models_ub.pdf", width=12, height=6)
plotCD(UB_df_plot, cex=1, decreasing=FALSE)
dev.off()


agg_record_297074224 
                   2

agg_record_297074224 
                   2

In [9]:
pairwise_win_count <- function(df, win_type = c("highest", "lowest"), approaches_to_compare = NULL) {
  win_type <- match.arg(win_type)
  # Select only the approaches to be compared, or all if approaches_to_compare not provided
  if (!is.null(approaches_to_compare)) {
    approaches <- intersect(approaches_to_compare, colnames(df))
    sub_df <- df[, approaches, drop = FALSE]
  } else {
    approaches <- colnames(df)
    sub_df <- df
  }
  n_approaches <- length(approaches)
  win_matrix <- matrix(0, nrow = n_approaches, ncol = n_approaches,
                       dimnames = list(approaches, approaches))
  
  for (i in 1:nrow(sub_df)) {
    row_vals <- as.numeric(sub_df[i, ])
    for (a in 1:n_approaches) {
      for (b in 1:n_approaches) {
        # Do not compare same approach and only compare non-NA pairs
        if (a != b && !is.na(row_vals[a]) && !is.na(row_vals[b])) {
          if (win_type == "highest" && (row_vals[a] > row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
          if (win_type == "lowest" && (row_vals[a] < row_vals[b])) {
            win_matrix[a, b] <- win_matrix[a, b] + 1
          }
        }
      }
    }
  }
  # Return the square matrix directly (not in melted data frame form)
  return(win_matrix)
}


In [10]:
# Example usage for LB (highest value is a win)
LB_pairwise_win_matrix <- pairwise_win_count(LB_df, win_type = "highest", approaches_to_compare = c("Walk-CBRP", "Walk-CBRP-Frac", "Walk-CBRP-Frac-Prep", "Walk-CBRP-Prep", "Walk-CBRP-MTZ"))
print("Pairwise win counts for LB (Higher is better):")
print(LB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
LB_pairwise_win_matrix_disp <- LB_pairwise_win_matrix
diag(LB_pairwise_win_matrix_disp) <- "-"
print("Pairwise win counts for LB (Higher is better) [with '-' on diagonal]:")
print(LB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
LB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(LB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
LB_pairwise_win_matrix_disp_df[] <- lapply(LB_pairwise_win_matrix_disp_df, as.character)
cat("Win counts Among Model LBs\n")
print(xtable(LB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model LBs.", 
             align = c("l", rep("c", ncol(LB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)

[1] "Pairwise win counts for LB (Higher is better):"
                    Walk-CBRP Walk-CBRP-Frac Walk-CBRP-Frac-Prep Walk-CBRP-Prep
Walk-CBRP                   0              0                  14             14
Walk-CBRP-Frac              0              0                  14             14
Walk-CBRP-Frac-Prep        17             17                   0              0
Walk-CBRP-Prep             17             17                   0              0
Walk-CBRP-MTZ              10             10                   3              3
                    Walk-CBRP-MTZ
Walk-CBRP                      27
Walk-CBRP-Frac                 27
Walk-CBRP-Frac-Prep            34
Walk-CBRP-Prep                 34
Walk-CBRP-MTZ                   0
[1] "Pairwise win counts for LB (Higher is better) [with '-' on diagonal]:"
                    Walk-CBRP Walk-CBRP-Frac Walk-CBRP-Frac-Prep Walk-CBRP-Prep
Walk-CBRP           "-"       "0"            "14"                "14"          
Walk-CBRP-Frac      "0"    

Win counts Among Model LBs
% latex table generated in R 4.5.2 by xtable 1.8-4 package
% Sat Jan 10 18:22:02 2026
\begin{table}[ht]
\centering
\begin{tabular}{lccccc}
  \hline
 & Walk-CBRP & Walk-CBRP-Frac & Walk-CBRP-Frac-Prep & Walk-CBRP-Prep & Walk-CBRP-MTZ \\ 
  \hline
Walk-CBRP & - & 0 & 14 & 14 & 27 \\ 
  Walk-CBRP-Frac & 0 & - & 14 & 14 & 27 \\ 
  Walk-CBRP-Frac-Prep & 17 & 17 & - & 0 & 34 \\ 
  Walk-CBRP-Prep & 17 & 17 & 0 & - & 34 \\ 
  Walk-CBRP-MTZ & 10 & 10 & 3 & 3 & - \\ 
   \hline
\end{tabular}
\caption{Win counts Among Model LBs.} 
\end{table}


In [26]:
# Example usage for LB (highest value is a win)
UB_pairwise_win_matrix <- pairwise_win_count(UB_df, win_type = "lowest", approaches_to_compare = c("Path-CBRP-MTZ", "Path-CBRP-MTZ-Prep", "Path-CBRP-Prep"))
print("Pairwise win counts for UB (Lower is better):")
print(UB_pairwise_win_matrix)

# Replace zeros on the diagonal with hyphen "-"
UB_pairwise_win_matrix_disp <- UB_pairwise_win_matrix
diag(UB_pairwise_win_matrix_disp) <- "-"
print("Pairwise win counts for LB (Lower is better) [with '-' on diagonal]:")
print(UB_pairwise_win_matrix_disp)

# Generate LaTeX table
library(xtable)
# Convert matrix with hyphens to data.frame for xtable compatibility
UB_pairwise_win_matrix_disp_df <- as.data.frame.matrix(UB_pairwise_win_matrix_disp)
# xtable will attempt to coerce numeric columns; force all to character to preserve "-"
UB_pairwise_win_matrix_disp_df[] <- lapply(UB_pairwise_win_matrix_disp_df, as.character)
cat("Win counts Among Model UBs\n")
print(xtable(UB_pairwise_win_matrix_disp_df, 
             caption = "Win counts Among Model UBs.", 
             align = c("l", rep("c", ncol(UB_pairwise_win_matrix_disp_df)))),
      include.rownames=TRUE, sanitize.text.function=identity)

[1] "Pairwise win counts for UB (Lower is better):"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ                  0                  2             14
Path-CBRP-MTZ-Prep            13                  0             19
Path-CBRP-Prep                 1                  0              0
[1] "Pairwise win counts for LB (Lower is better) [with '-' on diagonal]:"
                   Path-CBRP-MTZ Path-CBRP-MTZ-Prep Path-CBRP-Prep
Path-CBRP-MTZ      "-"           "2"                "14"          
Path-CBRP-MTZ-Prep "13"          "-"                "19"          
Path-CBRP-Prep     "1"           "0"                "-"           
Win counts Among Model UBs
% latex table generated in R 4.5.2 by xtable 1.8-4 package
% Thu Nov  6 21:14:50 2025
\begin{table}[ht]
\centering
\begin{tabular}{lccc}
  \hline
 & Path-CBRP-MTZ & Path-CBRP-MTZ-Prep & Path-CBRP-Prep \\ 
  \hline
Path-CBRP-MTZ & - & 2 & 14 \\ 
  Path-CBRP-MTZ-Prep & 13 & - & 19 \\ 
  Path-CBRP-Prep & 1 & 0 &

In [ ]:
# ============================================================================
# COMPREHENSIVE ANALYSIS FOR COMPUTATIONAL EXPERIMENTS SECTION
# ============================================================================

# Helper function to get runtime (handles different column names)
get_runtime <- function(df) {
  runtime_col <- intersect(c("Runtime (s)", "Time (s)"), colnames(df))
  if (length(runtime_col) > 0) return(df[[runtime_col[1]]])
  return(rep(NA, nrow(df)))
}

# Helper function to get attended blocks
get_attended <- function(df) {
  blocks_col <- intersect(c("Attended Blocks", "Attended"), colnames(df))
  if (length(blocks_col) > 0) return(df[[blocks_col[1]]])
  return(rep(NA, nrow(df)))
}

# ============================================================================
# COMPUTE KEY METRICS FOR EACH METHODOLOGY
# ============================================================================

compute_metrics <- function(df, method_name) {
  metrics <- list(method = method_name)
  metrics$n_instances <- nrow(df)
  
  # Optimal solutions (gap == 0)
  metrics$optimal_count <- sum(df[["gap (%)"]] == 0, na.rm = TRUE)
  metrics$optimal_pct <- round(100 * metrics$optimal_count / metrics$n_instances, 1)
  
  # Gap statistics
  metrics$avg_gap <- round(mean(df[["gap (%)"]], na.rm = TRUE), 4)
  metrics$max_gap <- round(max(df[["gap (%)"]], na.rm = TRUE), 4)
  
  # LB and UB
  metrics$avg_LB <- round(mean(df$LB, na.rm = TRUE), 2)
  metrics$avg_UB <- round(mean(df$UB, na.rm = TRUE), 2)
  
  # Runtime
  runtimes <- get_runtime(df)
  metrics$avg_runtime <- round(mean(runtimes, na.rm = TRUE), 2)
  metrics$max_runtime <- round(max(runtimes, na.rm = TRUE), 2)
  metrics$min_runtime <- round(min(runtimes, na.rm = TRUE), 4)
  
  # Attended blocks
  attended <- get_attended(df)
  metrics$avg_attended <- round(mean(attended, na.rm = TRUE), 1)
  
  # Problem size
  metrics$avg_V <- round(mean(as.numeric(df[["||V||"]]), na.rm = TRUE), 1)
  metrics$avg_A <- round(mean(as.numeric(df[["||A||"]]), na.rm = TRUE), 1)
  metrics$avg_B <- round(mean(as.numeric(df[["||B||"]]), na.rm = TRUE), 1)
  
  return(metrics)
}

# Compute metrics for all methodologies
all_metrics <- lapply(names(data_list), function(name) {
  compute_metrics(data_list[[name]], name)
})
names(all_metrics) <- names(data_list)

# ============================================================================
# DISPLAY SUMMARY TABLE
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("SUMMARY METRICS FOR ALL METHODOLOGIES\n")
cat("===============================================================================\n\n")

# Create summary dataframe
summary_df <- data.frame(
  Method = sapply(all_metrics, function(x) x$method),
  Inst = sapply(all_metrics, function(x) x$n_instances),
  Opt = sapply(all_metrics, function(x) x$optimal_count),
  Opt_Pct = sapply(all_metrics, function(x) paste0(x$optimal_pct, "%")),
  Avg_Gap = sapply(all_metrics, function(x) paste0(x$avg_gap, "%")),
  Avg_Time = sapply(all_metrics, function(x) x$avg_runtime),
  Avg_Attend = sapply(all_metrics, function(x) x$avg_attended),
  Avg_LB = sapply(all_metrics, function(x) x$avg_LB),
  Avg_UB = sapply(all_metrics, function(x) x$avg_UB),
  stringsAsFactors = FALSE
)
rownames(summary_df) <- NULL
print(summary_df)

# ============================================================================
# PREPROCESSING IMPACT ANALYSIS
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("PREPROCESSING IMPACT ANALYSIS\n")
cat("===============================================================================\n")

# Path-CBRP vs Path-CBRP-Prep
cat("\n--- Path-CBRP vs Path-CBRP-Prep (Exponential Formulation) ---\n")
exp <- data_list[["Path-CBRP"]]
exp_prep <- data_list[["Path-CBRP-Prep"]]

exp_V <- mean(as.numeric(exp[["||V||"]]), na.rm = TRUE)
exp_prep_V <- mean(as.numeric(exp_prep[["||V||"]]), na.rm = TRUE)
cat("Problem Size |V|:", round(exp_V, 1), "->", round(exp_prep_V, 1), 
    "(", round(100*(1-exp_prep_V/exp_V), 1), "% reduction)\n")

exp_time <- mean(get_runtime(exp), na.rm = TRUE)
exp_prep_time <- mean(get_runtime(exp_prep), na.rm = TRUE)
cat("Avg Runtime:", round(exp_time, 2), "s ->", round(exp_prep_time, 2), "s",
    "(", round(100*(1-exp_prep_time/exp_time), 1), "% reduction)\n")

exp_gap <- mean(exp[["gap (%)"]], na.rm = TRUE)
exp_prep_gap <- mean(exp_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(exp_gap, 4), "% ->", round(exp_prep_gap, 4), "%\n")

# Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep
cat("\n--- Path-CBRP-MTZ vs Path-CBRP-MTZ-Prep (MTZ Formulation) ---\n")
mtz <- data_list[["Path-CBRP-MTZ"]]
mtz_prep <- data_list[["Path-CBRP-MTZ-Prep"]]

mtz_time <- mean(get_runtime(mtz), na.rm = TRUE)
mtz_prep_time <- mean(get_runtime(mtz_prep), na.rm = TRUE)
cat("Avg Runtime:", round(mtz_time, 2), "s ->", round(mtz_prep_time, 2), "s",
    "(", round(100*(1-mtz_prep_time/mtz_time), 1), "% reduction)\n")

mtz_gap <- mean(mtz[["gap (%)"]], na.rm = TRUE)
mtz_prep_gap <- mean(mtz_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(mtz_gap, 4), "% ->", round(mtz_prep_gap, 4), "%\n")

# Path-CBRP-Frac vs Path-CBRP-Frac-Prep
cat("\n--- Path-CBRP-Frac vs Path-CBRP-Frac-Prep (Fractional Cuts) ---\n")
frac <- data_list[["Path-CBRP-Frac"]]
frac_prep <- data_list[["Path-CBRP-Frac-Prep"]]

frac_gap <- mean(frac[["gap (%)"]], na.rm = TRUE)
frac_prep_gap <- mean(frac_prep[["gap (%)"]], na.rm = TRUE)
cat("Avg Gap:", round(frac_gap, 4), "% ->", round(frac_prep_gap, 4), "% (WORSENED)\n")

frac_opt <- sum(frac[["gap (%)"]] == 0, na.rm = TRUE)
frac_prep_opt <- sum(frac_prep[["gap (%)"]] == 0, na.rm = TRUE)
cat("Optimal solutions:", frac_opt, "->", frac_prep_opt, "\n")

# ============================================================================
# GENERATE LATEX TABLE FOR PAPER
# ============================================================================

cat("\n")
cat("===============================================================================\n")
cat("LATEX TABLE - COMPUTATIONAL RESULTS\n")
cat("===============================================================================\n\n")

library(xtable)
latex_df <- summary_df
colnames(latex_df) <- c("Method", "Inst.", "Opt.", "Opt.(%)", "Avg.Gap(%)", 
                         "Avg.Time(s)", "Avg.Blocks", "Avg.LB", "Avg.UB")
print(xtable(latex_df, 
             caption = "Summary of computational results for all methodologies.", 
             label = "tab:summary_results",
             align = c("l", "l", rep("c", 8))),
      include.rownames = FALSE,
      sanitize.text.function = identity)